# 🔱 VoiceBatch Studio v2.4.0 - [Chatterbox Official]
GitHub Repository के अनुसार सटीक इंस्टॉलेशन और हिंदी सपोर्ट।

In [ ]:
# @title 💤 Step 1: Official Installation (Source Build)
import os
from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ Chatterbox को GitHub से क्लोन और इंस्टॉल किया जा रहा है...")
# GitHub Repo के अनुसार इंस्टॉलेशन
!git clone https://github.com/resemble-ai/chatterbox.git
%cd chatterbox
!pip install -e .
!pip install -q gradio librosa soundfile coqui-tts torchaudio perth
!python -m spacy download ja_core_news_sm

os.makedirs("/content/outputs", exist_ok=True)
print("✅ इंस्टॉलेशन पूरा हुआ! अब Step 2 चलाएं।")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (XTTS + Chatterbox Turbo)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf, torchaudio as ta
from TTS.api import TTS
from chatterbox.tts_turbo import ChatterboxTurboTTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# लोड हो रहे हैं मॉडल्स
print("⏳ मॉडल्स लोड हो रहे हैं...")
xtts_model = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
chatter_model = ChatterboxTurboTTS.from_pretrained(device=device)

def strict_hindi_filter(text):
    pattern = re.compile(r'[^\u0900-\u097F\s।,?!:;0-9\[\]]')
    return pattern.sub('', text)

def studio_pro_engine(engine_type, text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    if lang == 'hi': text = strict_hindi_filter(text)
    
    out_path = '/content/outputs/VoiceBatch_Studio_Output.wav'
    
    if engine_type == 'XTTS v2 (Stable)':
        # XTTS Engine
        xtts_model.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=out_path, split_sentences=True)
        y, sr = librosa.load(out_path)
    else:
        # Chatterbox Turbo Engine (Official Logic)
        # GitHub सुझाव: exaggeration=0.5, cfg_weight=0.5
        wav = chatter_model.generate(text, audio_prompt_path=audio_sample)
        # ta.save का उपयोग करके सेव करना जैसा GitHub पर बताया गया है
        ta.save(out_path, wav.cpu(), chatter_model.sr)
        y, sr = librosa.load(out_path)
        
    # पोस्ट प्रोसेसिंग (Speed, Pitch, Silence)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.4.0')
    with gr.Row():
        with gr.Column():
            engine = gr.Radio(['XTTS v2 (Stable)', 'Chatterbox Turbo (Fast)'], label='Select Engine', value='XTTS v2 (Stable)')
            txt = gr.Textbox(label='Hindi Script', lines=10, placeholder='Chatterbox के लिए [chuckle] जैसे टैग्स काम करेंगे।')
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, label="Speed")
                ptc = gr.Slider(-4, 4, 0, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate High Quality Audio 🔱', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download Output')
            gr.Markdown('### Chatterbox Tips:\n- [chuckle] [laugh] [sigh] टैग्स इस्तेमाल करें।\n- हिंदी कहानी के लिए XTTS ज्यादा स्टेबल है।')

    btn.click(studio_pro_engine, [engine, txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True, debug=True)
'''
with open('/content/app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है! अब लॉन्च हो रहा है...")
!python /content/app.py